# 02 — Quality Assessment

Runs the full check registry in `src/dq_checks.py`, renders figures via `src/report.py`, and validates a sample against the executable intake schema in `src/intake_schema.py`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from load import load
from dq_checks import run_all
from report import missingness_bar, price_distribution, severity_breakdown

df = load()
dq = run_all(df)
dq

,Dimension,Check,Failing rows,% of total,Severity,Pricing-model impact
0,Validity,Currency column absent,1067371,100.00%,Blocker,"Without an explicit currency, any multi-countr..."
1,Completeness,Customer ID is null,243007,22.77%,Blocker,Customer-level price elasticity and retention ...
2,Completeness,Price is null,0,0.00%,Blocker,Price is the target variable for a pricing mod...
3,Uniqueness,Exact duplicate rows,12133,1.14%,High,Duplicate rows inflate both revenue and volume...
4,Validity,Price <= 0,6207,0.58%,High,Zero or negative unit prices contaminate the p...
5,Anomaly,"Non-product stock codes (POST, DOT, BANK CHARG...",5810,0.54%,High,Adjustments and fees share the transactional s...
6,Integrity,Negative quantity on non-cancellation invoice,3457,0.32%,High,Negative quantities outside the cancellation c...
7,Integrity,Cancellation rows with no matching original pu...,2234,0.21%,High,Unmatched cancellations either inflate returns...
8,Consistency,StockCodes with multiple distinct descriptions,420566,39.40%,Medium,Description drift on the same SKU signals upst...
9,Uniqueness,"Duplicate (Invoice, StockCode) pairs",88585,8.30%,Medium,Multiple lines of the same SKU on the same inv...


In [2]:
dq.to_csv('../output/dq_results.csv', index=False)
print(dq.to_markdown(index=False))

| Dimension    | Check                                                         |   Failing rows | % of total   | Severity   | Pricing-model impact                                                                                                                                  |
|:-------------|:--------------------------------------------------------------|---------------:|:-------------|:-----------|:------------------------------------------------------------------------------------------------------------------------------------------------------|
| Validity     | Currency column absent                                        |        1067371 | 100.00%      | Blocker    | Without an explicit currency, any multi-country pricing model silently mixes GBP, EUR, and other currencies as if comparable.                         |
| Completeness | Customer ID is null                                           |         243007 | 22.77%       | Blocker    | Customer-level price elasticity and rete

In [3]:
missingness_bar(df)
price_distribution(df)
severity_breakdown(dq)

WindowsPath('D:/pryzm_task/output/figures/severity_breakdown.png')

In [4]:
# Prove the intake schema is executable: valid row passes, raw-style row fails
import pandas as pd
from intake_schema import validate

good = pd.DataFrame([{
    'invoice_id': 'INV-0001', 'line_id': '1', 'sku_id': 'SKU-22423',
    'product_description': 'WHITE HANGING HEART T-LIGHT HOLDER',
    'quantity': 6, 'unit_price': 2.55, 'currency': 'GBP',
    'invoice_datetime': pd.Timestamp('2010-12-01T08:26:00'),
    'customer_id': '17850', 'country': 'GB',
    'is_cancellation': False, 'original_invoice_id': None,
}])
bad = pd.DataFrame([{
    'invoice_id': '536365', 'line_id': '1', 'sku_id': '85123a',
    'product_description': 'WHITE HANGING HEART T-LIGHT HOLDER',
    'quantity': 0, 'unit_price': -1.5, 'currency': 'gbp',
    'invoice_datetime': pd.Timestamp('2010-12-01T08:26:00'),
    'customer_id': '17850', 'country': 'United Kingdom',
    'is_cancellation': False, 'original_invoice_id': None,
}])
print('VALID:', validate(good)[0])
print('RAW  :', validate(bad)[0])

VALID: True
RAW  : False


D:\pryzm_task\.venv\Lib\site-packages\pandera\_pandas_deprecated.py:143: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)
